In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import scipy.sparse as sp
import doubletdetection as dd

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/ouptuts/h5ad_files/inprogress.h5ad')

In [ ]:
adata

In [ ]:
# choose parameters here
DD_PARAMS = dict(
    n_iters=50,
    clustering_algorithm="louvain",  # or 'leiden'
    standard_scaling=True,
    pseudocount=0.1,
    n_jobs=8,
    random_state=0,
)
P_THRESH   = 1e-16   # very stringent; adjust as needed
VOTER_THRS = 0.5

# storage
dd_score  = pd.Series(index=adata.obs_names, dtype="float64")
dd_pred   = pd.Series(index=adata.obs_names, dtype="boolean")

for b in adata.obs["batch"].unique():
    idx = adata.obs["batch"] == b
    ad_b = adata[idx].copy()

    # get raw counts
    X = ad_b.layers["counts"]
    if sp.issparse(X):
        X = X.tocsr()  # DoubletDetection can take sparse CSR or dense
    # fit/predict
    clf = dd.BoostClassifier(**DD_PARAMS)
    doublets = clf.fit(X).predict(p_thresh=P_THRESH, voter_thresh=VOTER_THRS)

    # collect
    dd_score.loc[ad_b.obs_names] = clf.doublet_score()
    dd_pred.loc[ad_b.obs_names]  = pd.Series(doublets, index=ad_b.obs_names).astype("boolean")

# write to obs
adata.obs["dd_score"] = dd_score
adata.obs["dd_is_doublet"] = dd_pred
adata.uns["doubletdetection_params"] = {**DD_PARAMS, "p_thresh": P_THRESH, "voter_thresh": VOTER_THRS}
print("DoubletDetection done:", int(adata.obs["dd_is_doublet"].sum()), "doublets flagged.")

In [ ]:
# --- Scrublet ---
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scrublet as scr

# Parameters
SCR_PARAMS = dict(
    expected_doublet_rate=0.05,
    sim_doublet_ratio=2.0,
    random_state=0,
)
N_PRIN_COMPS = 30

scr_score = pd.Series(index=adata.obs_names, dtype="float64")
scr_pred  = pd.Series(index=adata.obs_names, dtype="boolean")
scr_thrs  = pd.Series(index=adata.obs_names, dtype="float64")

for b in adata.obs["batch"].unique():
    idx = adata.obs["batch"] == b
    ad_b = adata[idx].copy()

    # raw counts
    X = ad_b.layers["counts"]
    if sp.issparse(X):
        X = X.tocsr()
    if sp.issparse(X):
        X = X.toarray()  # Scrublet needs dense

    scrub = scr.Scrublet(X, **SCR_PARAMS)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(
        min_counts=2,
        min_cells=3,
        use_approx_neighbors=True,
        n_prin_comps=N_PRIN_COMPS,
    )

    # store results
    scr_score.loc[ad_b.obs_names] = doublet_scores
    scr_pred.loc[ad_b.obs_names]  = pd.Series(predicted_doublets, index=ad_b.obs_names).astype("boolean")
    scr_thrs.loc[ad_b.obs_names]  = float(scrub.threshold_)

adata.obs["scrublet_score"] = scr_score
adata.obs["scrublet_is_doublet"] = scr_pred
adata.obs["scrublet_threshold_used"] = scr_thrs
adata.uns["scrublet_params"] = {**SCR_PARAMS, "n_prin_comps": N_PRIN_COMPS}

print("Scrublet done:", int(adata.obs["scrublet_is_doublet"].sum()), "doublets flagged.")


In [ ]:
adata.write_h5ad(f'{ACC_DATA_ROOT}/ouptuts/h5ad_files/inprogress_dd_scr.h5ad', compression = 'lzf')